# Tworzymy model do klastrowania danych z ankiety powitalnej

In [19]:
import pandas as pd
from pycaret.clustering import setup, create_model, assign_model, plot_model, save_model, load_model, predict_model

In [22]:
df = pd.read_csv('welcome_survey_simple_v2.csv', sep=';')
df.fillna('Brak danych', inplace=True)
df.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,Brak danych,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


In [23]:
for col in df.columns:
    print(f'{col}: {df[col].unique()}')

age: ['<18' '25-34' '45-54' '35-44' '18-24' '>=65' '55-64' 'unknown']
edu_level: ['Podstawowe' 'Średnie' 'Wyższe']
fav_animals: ['Brak ulubionych' 'Psy' 'Koty' 'Inne' 'Koty i Psy']
fav_place: ['Brak danych' 'Nad wodą' 'W lesie' 'W górach' 'Inne']
gender: ['Kobieta' 'Mężczyzna' 'Brak danych']


In [4]:
len(df)

229

In [24]:
s = setup(df, session_id=123)
s

,Description,Value
0,Session id,123
1,Original data shape,"(229, 5)"
2,Transformed data shape,"(229, 24)"
3,Categorical features,5
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Maximum one-hot encoding,-1
9,Encoding method,None


In [25]:
s.dataset.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,Brak danych,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


In [26]:
s.dataset_transformed.head()

,age_<18,age_25-34,age_45-54,age_35-44,age_18-24,age_>=65,age_55-64,age_unknown,edu_level_Podstawowe,edu_level_Średnie,...,fav_animals_Inne,fav_animals_Koty i Psy,fav_place_Brak danych,fav_place_Nad wodą,fav_place_W lesie,fav_place_W górach,fav_place_Inne,gender_Kobieta,gender_Mężczyzna,gender_Brak danych
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [27]:
kmeans = create_model('kmeans', num_clusters=8)


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.1525,21.2510,1.9273,0,0,0


In [28]:
df_with_clusters = assign_model(kmeans)
df_with_clusters

,age,edu_level,fav_animals,fav_place,gender,Cluster
0,<18,Podstawowe,Brak ulubionych,Brak danych,Kobieta,Cluster 1
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna,Cluster 3
2,45-54,Wyższe,Psy,W lesie,Mężczyzna,Cluster 2
3,35-44,Średnie,Koty,W górach,Mężczyzna,Cluster 3
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna,Cluster 0
...,...,...,...,...,...,...
224,35-44,Wyższe,Koty,Inne,Kobieta,Cluster 7
225,45-54,Wyższe,Inne,W lesie,Mężczyzna,Cluster 2
226,25-34,Wyższe,Psy,W górach,Mężczyzna,Cluster 6
227,35-44,Wyższe,Brak ulubionych,W górach,Mężczyzna,Cluster 7


In [30]:
df_with_clusters["Cluster"].value_counts()

Cluster 0    42
Cluster 3    35
Cluster 2    33
Cluster 1    29
Cluster 5    26
Cluster 6    25
Cluster 7    24
Cluster 4    15
Name: Cluster, dtype: int64

In [31]:
plot_model(kmeans, plot='cluster')

In [32]:
save_model(kmeans, 'welcome_survey_clustering_pipeline_v2', verbose=False)

(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=[], transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                              'fav_place', 'gender'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('onehot_encoding',
                  TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                              'fav_place', 'gender'],
                                     transformer=OneHotEncoder(cols=['age',
                                                                     'edu_level',
                                                                     'fav_animals',
                                                                     'fav_place',
                                             

In [33]:
kmeans_pipeline = load_model('welcome_survey_clustering_pipeline_v2')
kmeans_pipeline

Transformation Pipeline and Model Successfully Loaded


Pipeline(memory=FastMemory(location=C:\Users\Win\AppData\Local\Temp\joblib),
         steps=[('numerical_imputer',
                 TransformerWrapper(include=[], transformer=SimpleImputer())),
                ('categorical_imputer',
                 TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                             'fav_place', 'gender'],
                                    transformer=SimpleImputer(strategy='most_frequent'))),
                ('onehot_encoding',
                 TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                             'fav_place', 'gender'],
                                    transformer=OneHotEncoder(cols=['age',
                                                                    'edu_level',
                                                                    'fav_animals',
                                                                    'fav_place',
                                                                    'gender'],
                                                              handle_missing='return_nan',
                                                              use_cat_names=True))),
                ('trained_model', KMeans(random_state=123))])

In [34]:
predict_df = pd.DataFrame([
    {
        "age": "45-54",  # '<18', '25-34', '45-54', '35-44', '18-24', '>=65', '55-64', 'unknown'
        "edu_level": 'Średnie',  # 'Podstawowe', 'Średnie', 'Wyższe'
        "fav_animals": 'Brak ulubionych',  # 'Brak ulubionych', 'Psy', 'Koty', 'Inne', 'Koty i Psy'
        "fav_place": 'W lesie',  # 'Nad wodą', 'W lesie', 'W górach', 'Inne'
        "gender": 'Kobieta',  # 'Mężczyzna', 'Kobieta'
    }
])

In [35]:
predict_with_clusters_df = predict_model(kmeans_pipeline, data=predict_df)
predict_with_clusters_df["Cluster"]

0    Cluster 1
Name: Cluster, dtype: object